# TAT-C Coverage Example

## Interface

Specifies a function with well-defined input (`CoverageRequest`) and output (`CoverageResponse`) using TAT-C's implementation.

In [ ]:
from datetime import timedelta
from joblib import Parallel, delayed
from scipy.stats import hmean
import geopandas as gpd
import pandas as pd

from tatc.schemas import (
    Instrument as TATC_Instrument,
    Satellite as TATC_Satellite,
    TwoLineElements,
    Point,
)
from tatc.analysis import (
    collect_multi_observations,
    aggregate_observations,
    reduce_observations,
)

from eose.coverage import (
    CoverageSample,
    CoverageRecord,
    CoverageRequest,
    CoverageResponse,
)
from eose.access import (
    AccessSample,
    AccessRecord,
    AccessRequest,
    AccessResponse,
)
from eose.grids import UniformAngularGrid
from eose.orbits import GeneralPerturbationsOrbitState, Propagator
from eose.satellites import Satellite, Payload


def access_tatc(request: AccessRequest) -> AccessResponse:
    if request.propagator != Propagator.SGP4:
        raise RuntimeError("TAT-C only supports SGP4 propagator.")
    satellites = [
        TATC_Satellite(
            name=satellite.id,
            orbit=TwoLineElements(tle=satellite.orbit.to_tle()),
            instruments=[
                TATC_Instrument(
                    name=str(payload.id), field_of_regard=payload.field_of_view
                )
            ],
        )
        for satellite in request.satellites
        for payload in satellite.payloads
        if payload.id in request.payload_ids
    ]
    observations = Parallel(-1)(
        delayed(collect_multi_observations)(
            Point(
                id=i,
                longitude=target.position[0],
                latitude=target.position[1],
                altitude=(target.position[2] if len(target.position) > 2 else 0),
            ),
            satellites,
            request.start,
            request.start + request.duration,
        )
        for i, target in enumerate(request.targets)
    )
    return AccessResponse(
        **request.model_dump(exclude="target_records"),
        target_records=[
            (
                AccessRecord(target_id=target.id)
                if observations[i].empty
                else AccessRecord(
                    target_id=target.id,
                    samples=observations[i].apply(
                        lambda s: AccessSample(
                            start=s.start,
                            duration=s.end - s.start,
                            satellite_id=s.satellite,
                            instrument_id=s.instrument,
                        ),
                        axis=1,
                    ),
                )
            )
            for i, target in enumerate(request.targets)
        ],
    )


def coverage_tatc(request: CoverageRequest) -> CoverageResponse:
    aggregated_obs = aggregate_observations(
        gpd.GeoDataFrame(
            [
                {
                    "point_id": request.targets.index(target),
                    "geometry": target.as_geometry(),
                    "satellite": sample.satellite_id,
                    "instrument": sample.instrument_id,
                    "start": sample.start,
                    "end": sample.start + sample.duration,
                    "epoch": sample.start + sample.duration / 2,
                }
                for record in request.target_records
                for target in [t for t in request.targets if t.id == record.target_id]
                for sample in record.samples
                if sample.satellite_id not in request.omit_satellite_ids
                if sample.instrument_id not in request.omit_payload_ids
            ]
        )
    )
    reduced_obs = reduce_observations(aggregated_obs)
    return CoverageResponse(
        **request.model_dump(
            exclude=["target_records", "harmonic_mean_revisit", "coverage_fraction"]
        ),
        target_records=list(
            reduced_obs.apply(
                lambda r: CoverageRecord(
                    **next(
                        record
                        for record in request.target_records
                        if request.targets[r["point_id"]].id == record.target_id
                    ).model_dump(exclude=["samples", "mean_revisit", "number_samples"]),
                    samples=aggregated_obs[
                        aggregated_obs.point_id == r["point_id"]
                    ].apply(
                        lambda s: CoverageSample(
                            start=s.start,
                            duration=s.end - s.start,
                            satellite_id=s.satellite,
                            instrument_id=s.instrument,
                            revisit=None if pd.isna(s.revisit) else s.revisit,
                        ),
                        axis=1,
                    ),
                    mean_revisit=(
                        None
                        if pd.isna(r["revisit"])
                        else timedelta(seconds=r["revisit"].total_seconds())
                    ),
                    number_samples=r["samples"],
                ),
                axis=1,
            )
        )
        + [
            CoverageRecord(target_id=target.id)
            for i, target in enumerate(request.targets)
            if not any(reduced_obs["point_id"] == i)
        ],
        harmonic_mean_revisit=(
            None
            if reduced_obs.dropna(subset="revisit").empty
            else timedelta(
                seconds=hmean(
                    reduced_obs.dropna(subset="revisit")["revisit"].dt.total_seconds()
                )
            )
        ),
        coverage_fraction=len(reduced_obs.index) / len(request.targets),
    )

## Demonstration

Pulls OMM file from Celestrak, issues observation request starting Jan 1 2024 for 7 days), and displays results.

In [ ]:
import json
from datetime import datetime, timedelta, timezone

# note: celestrak request is rate-limited; using hard-coded version
# import requests
# response = requests.get("https://celestrak.org/NORAD/elements/gp.php?NAME=ZARYA&FORMAT=JSON").content
response = '[{"OBJECT_NAME":"ISS (ZARYA)","OBJECT_ID":"1998-067A","EPOCH":"2024-06-07T09:53:34.728000","MEAN_MOTION":15.50975122,"ECCENTRICITY":0.0005669,"INCLINATION":51.6419,"RA_OF_ASC_NODE":3.7199,"ARG_OF_PERICENTER":284.672,"MEAN_ANOMALY":139.0837,"EPHEMERIS_TYPE":0,"CLASSIFICATION_TYPE":"U","NORAD_CAT_ID":25544,"ELEMENT_SET_NO":999,"REV_AT_EPOCH":45703,"BSTAR":0.00033759,"MEAN_MOTION_DOT":0.00019541,"MEAN_MOTION_DDOT":0}]'
iss_omm = json.loads(response)[0]

In [ ]:
from shapely.geometry import box, mapping

request = AccessRequest(
    satellites=[
        Satellite(
            id="ISS",
            orbit=GeneralPerturbationsOrbitState.from_omm(iss_omm),
            payloads=[Payload(id="Test", field_of_view=100)],
        )
    ],
    targets=UniformAngularGrid(
        delta_latitude=20, delta_longitude=20, region=mapping(box(-180, -50, 180, 50))
    ).as_targets(),
    start=datetime(2024, 1, 1, tzinfo=timezone.utc),
    duration=timedelta(days=7),
    propagator=Propagator.SGP4,
    payload_ids=["Test"],
)

display(request.model_dump_json())

response = access_tatc(request)

display(response.model_dump_json())

data = response.as_dataframe()

display(data)

Append coverage analysis to observation results.

In [ ]:
request2 = CoverageRequest(**response.model_dump())

display(request2.model_dump_json())

response2 = coverage_tatc(request2)

display(response2.model_dump_json())

data2 = response2.as_dataframe()

display(data2)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# example composite plot using GeoDataFrames
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_title(f"Number Samples (Coverage={response2.coverage_fraction:.1%})")
data2.plot(ax=ax, column="number_samples", legend=True)
ax.set_global()
ax.coastlines()
plt.show()

# example composite plot using GeoDataFrames
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})
ax.set_title(
    f"Mean Revisit (Harmonic Mean={response2.harmonic_mean_revisit/timedelta(hours=1):.1f} hr)"
)
data2["mean_revisit_hr"] = data2.apply(
    lambda r: r["mean_revisit"] / timedelta(hours=1), axis=1
)
data2.plot(ax=ax, column="mean_revisit_hr", legend=True)
ax.set_global()
ax.coastlines()
plt.show()